# Multi-Query RAG Pipeline
### PDF → Chunking → FAISS → Query Expansion → Parallel Search → Merge → Answer

In [ ]:
!pip install langchain langchain-community langchain-ollama langchain-text-splitters faiss-cpu pypdf -q

In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings, ChatOllama

c:\Users\shiva\.pyenv\pyenv-win\versions\3.12.10\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 1: Load PDF

In [2]:
loader = PyPDFLoader("A._P._J._Abdul_Kalam.pdf")
pages = loader.load()

print(f"Total Pages: {len(pages)}")
print(f"Sample Content:\n{pages[0].page_content[:500]}")

Total Pages: 30
Sample Content:
A. P. J. Abdul Kalam
Official portrait, c. 2002
President of India
In office
25 July 2002 – 25 July 2007
Prime Minister Atal Bihari Vajpayee
Manmohan Singh
Vice President Krishan Kant
Bhairon Singh Shekhawat
Preceded by K. R. Narayanan
Succeeded by Pratibha Patil
Principal Scientific Adviser to the
Government of India
In office
November 1999 – November 2001
President K. R. Narayanan
Prime Minister Atal Bihari Vajpayee
Preceded by Office established
Succeeded by Rajagopala Chidambaram
Director Ge


## Step 2: Split into Chunks

In [3]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(pages)

print(f"Total Chunks: {len(chunks)}")

Total Chunks: 136


## Step 3: Create Embeddings & Store in FAISS

In [4]:
embeddings = OllamaEmbeddings(model="nomic-embed-text:latest")

vector_store = FAISS.from_documents(chunks, embeddings)

print(f"Vector Store Created with {vector_store.index.ntotal} vectors")

Vector Store Created with 136 vectors


## Step 4: Create Retriever

In [5]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

## Step 5: Generate Query Variations using LLM

In [6]:
llm = ChatOllama(model="llama3.2:3b", temperature=0)

query = "What contributions did Abdul Kalam make to India's missile program?"

expand_prompt = f"""Generate 3 different versions of the following question to help retrieve 
relevant documents from a vector database. Each version should approach the question 
from a different angle. Return only the 3 questions, one per line.

Original question: {query}"""

response = llm.invoke(expand_prompt)
query_variations = [q.strip() for q in response.content.strip().split("\n") if q.strip()]

print("Original Query:", query)
print("\nGenerated Variations:")
for i, q in enumerate(query_variations, 1):
    print(f"  {i}. {q}")

Original Query: What contributions did Abdul Kalam make to India's missile program?

Generated Variations:
  1. 1. What was the role of Abdul Kalam in India's defense industry?
  2. 2. How did Abdul Kalam's work impact India's national security?
  3. 3. What notable achievements did Abdul Kalam make in the field of space exploration?


## Step 6: Retrieve for Each Variation & Merge Results

In [7]:
all_queries = [query] + query_variations

all_docs = []
for q in all_queries:
    docs = retriever.invoke(q)
    all_docs.extend(docs)

# Deduplicate by page content
seen = set()
unique_docs = []
for doc in all_docs:
    if doc.page_content not in seen:
        seen.add(doc.page_content)
        unique_docs.append(doc)

print(f"Total Retrieved: {len(all_docs)}")
print(f"After Deduplication: {len(unique_docs)}")

Total Retrieved: 12
After Deduplication: 6


In [10]:
print(all_docs[1])

page_content='2002 to 2007.
Born and raised in a Muslim family in Rameswaram,
Tamil Nadu, Kalam studied physics and aerospace
engineering. He spent the next four decades as a
scientist and science administrator, mainly at the
Defence Research and Development Organisation
(DRDO) and Indian Space Research Organisation
(ISRO) and was intimately involved in India's civilian
space programme and military missile development
efforts. He was known as the "Missile Man of India"
for his work on the development of ballistic missile
and launch vehicle technology. He also played a
pivotal organisational, technical, and political role in
Pokhran-II nuclear tests in 1998, India's second such
test after the first test in 1974.
Kalam was elected as the president of India in 2002
with the support of both the ruling Bharatiya Janata
Party and the then-opposition Indian National
Congress. He was widely referred to as the "People's
President". He engaged in teaching, writing and public' metadata={'producer

## Step 7: Generate Final Answer

In [11]:
context = "\n\n".join([doc.page_content for doc in unique_docs])

prompt = f"""Answer the question based only on the following context:

{context}

Question: {query}
Answer:"""

response = llm.invoke(prompt)
print("Answer:", response.content)

Answer: According to the text, Abdul Kalam made the following contributions to India's missile program:

* He played a major role in the development of missiles, including the Agni and Prithvi missiles.
* He was instrumental in convincing the cabinet to allocate ₹3.88 billion (equivalent to US$780 million in 2023) for the Integrated Guided Missile Development Programme (IGMDP) and was appointed as its chief executive.
* He worked on the development of a low-cost coronary stent, named the "Kalam-Raju stent".
* He developed a tablet computer named the "Kalam-Raju tablet" for usage by healthcare workers in rural areas.
* He played a pivotal organisational, technical, and political role in the Pokhran-II nuclear tests conducted in May 1998, India's second nuclear test after the first test in 1974.

Note that the text does not provide a comprehensive list of Kalam's contributions to the missile program, but rather highlights some of his notable achievements in this area.


## Retrieved Source Chunks

In [9]:
for i, doc in enumerate(unique_docs):
    print(f"\n--- Chunk {i+1} (Page {doc.metadata['page'] + 1}) ---")
    print(doc.page_content[:300])


--- Chunk 1 (Page 4) ---
US$780 million in 2023) for the project titled Integrated Guided Missile Development Programme
(IGMDP) and appointed Kalam as its chief executive.[28] Kalam played a major role in the development
of missiles including Agni, an intermediate range ballistic missile and Prithvi, the tactical surface-to

--- Chunk 2 (Page 1) ---
2002 to 2007.
Born and raised in a Muslim family in Rameswaram,
Tamil Nadu, Kalam studied physics and aerospace
engineering. He spent the next four decades as a
scientist and science administrator, mainly at the
Defence Research and Development Organisation
(DRDO) and Indian Space Research Organisat

--- Chunk 3 (Page 4) ---
Kalam greeting then prime minister
Vajpayee on 25 December 2003
major role in convincing the cabinet to conceal the true nature of these classified projects. His research
and leadership brought him recognition in the 1980s, which prompted the government to initiate an
advanced missile programme unde

--- Chunk 4 (Page 

## Test Questions

1. What year was Abdul Kalam born?
2. What was the name of the coronary stent Kalam co-developed?
3. Which institution did Kalam study aerospace engineering at?
4. How many electoral votes did Kalam receive in the 2002 presidential election?
5. What was Kalam's role at DRDO from 1992 to 1999?